# LangExtract — Quick Exploration
Testing LangExtract on German/English job postings using local Ollama (Llama 3.1 8B).  
No API key needed.

**Goal:** See how LangExtract extracts SKILL and TOOL entities, and compare feel vs. our fine-tuned model.

## 1. Install

In [1]:
# Run once
!pip install langextract --break-system-packages -q

In [62]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from project root

# LangExtract's Google Gemini provider reads GOOGLE_API_KEY
os.environ["GOOGLE_API_KEY"] = os.environ["LANGEXTRACT_API_KEY"]

print("✅ API key loaded")

✅ API key loaded


## 2. Imports + Verify Ollama is running

In [63]:
import requests
import langextract as lx
from langextract.providers.ollama import OllamaLanguageModel

OLLAMA_URL = "http://host.docker.internal:11434"

# Check Ollama is running
try:
    r = requests.get(OLLAMA_URL)
    print("✅ Ollama is running")
except Exception:
    print("❌ Ollama is NOT running — start it with: ollama serve")

# Create the model with the correct URL
ollama_model = OllamaLanguageModel(
    model_id="llama3.1:8b",
    model_url=OLLAMA_URL,
)

# Then pass it to extract() via the model parameter
# result = lx.extract(..., model=ollama_model, ...)


✅ Ollama is running


## 3. Sample Job Postings
One English, one German — same mix as your dataset.

In [49]:
english_job = """
Requirements:
3+ years of experience in machine learning and data science.
Strong proficiency in Python and experience with PyTorch or TensorFlow.
Experience with cloud platforms (AWS, GCP, or Azure).
Familiarity with MLOps practices including Docker, Kubernetes, and CI/CD pipelines.
Knowledge of NLP techniques and transformer-based models (e.g. BERT, GPT).
Experience with SQL and data engineering concepts.
Knowledge of statistical modeling and A/B testing.
"""

german_job = """
Anforderungen:
Mehrjährige Erfahrung in Machine Learning und Datenanalyse.
Sehr gute Kenntnisse in Python sowie Erfahrung mit PyTorch oder TensorFlow.
Erfahrung mit Cloud-Plattformen (AWS, Azure oder GCP).
Kenntnisse in MLOps und Deployment-Tools wie Docker und Kubernetes.
Erfahrung mit NLP und Transformer-Modellen.
Gute SQL-Kenntnisse und Verständnis von Datenpipelines.
Kenntnisse in statistischer Modellierung und A/B-Testing.
"""
description = """
Dein Skillset:\nAbschluss in Informatik, im Wirtschaftsingenieurswesen oder vergleichbar\nErfahrung in Automatisierung von ML-Pipelines, Plattformbetrieb und Orchestrierung mit Kubernetes/OpenShift sowie in der Optimierung von Monitoring- und Logging-Lösungen\nKenntnisse in Infrastructure-as-Code (Ansible, Terraform), CI/CD-Prozessen, Versionsverwaltung (Git) und GPU-Integration (z. B. NVIDIA) für ressourcenintensive KI-Workloads\nSicherer Umgang mit Tools wie MLflow, Kubeflow, Airflow sowie Grafana und Kibana für Monitoring und Logging; Erfahrung mit Containerisierung und Virtualisierungstechnologien\nDeutsch und Englisch sehr gut in Wort und Schrift
"""

print("Job postings ready.")

Job postings ready.


## 4. Define Prompt + Few-Shot Examples
This is the core of LangExtract — a natural language prompt + 2 examples.

In [50]:
prompt = textwrap.dedent("""
    Extract SKILL and TOOL entities from job posting requirements sections.
    
    SKILL: A technical competency, methodology, or domain knowledge.
    Examples: machine learning, NLP, deep learning, statistical modeling, MLOps, data engineering, CI/CD
    
    TOOL: A specific technology, programming language, framework, platform, or library.
    Examples: Python, PyTorch, TensorFlow, Docker, AWS, SQL, Kubernetes, scikit-learn
    
    Rules:
    - Use EXACT text from the source. Do not translate or paraphrase.
    - Do not extract soft skills (teamwork, communication).
    - Do not extract job titles or degree names.
    - Works in both German and English.
""")

examples = [
    lx.data.ExampleData(
        text="Experience with Python and machine learning. Familiarity with Docker and AWS.",
        extractions=[
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Python"),
            lx.data.Extraction(extraction_class="SKILL", extraction_text="machine learning"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Docker"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="AWS"),
        ]
    ),
    lx.data.ExampleData(
        text="Kenntnisse in Deep Learning und Datenanalyse. Erfahrung mit PyTorch und Kubernetes.",
        extractions=[
            lx.data.Extraction(extraction_class="SKILL", extraction_text="Deep Learning"),
            lx.data.Extraction(extraction_class="SKILL", extraction_text="Datenanalyse"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="PyTorch"),
            lx.data.Extraction(extraction_class="TOOL", extraction_text="Kubernetes"),
        ]
    ),
]

print("Prompt and examples ready.")

Prompt and examples ready.


## 5. Run Extraction — English Job

In [26]:
print("Running LangExtract on English job posting...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_en = lx.extract(
    text_or_documents=english_job,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434", 
    fence_output=False,
    use_schema_constraints=False,
)


print("=== ENGLISH JOB — Extracted Entities ===")
for extraction in result_en.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

Running LangExtract on English job posting...
(This may take 30-60 seconds with Llama 3.1 8B)



LangExtract: Processing, current=461 chars, processed=0 chars:  [00:30]

=== ENGLISH JOB — Extracted Entities ===
  [SKILL] 'machine learning'
  [SKILL] 'data science'
  [TOOL] 'Python'
  [TOOL] 'PyTorch'
  [TOOL] 'TensorFlow'
  [TOOL] 'AWS'
  [TOOL] 'GCP'
  [TOOL] 'Azure'
  [TOOL] 'Docker'
  [TOOL] 'Kubernetes'
  [TOOL] 'CI/CD pipelines'
  [SKILL] 'NLP techniques'
  [SKILL] 'transformer-based models'
  [TOOL] 'BERT'
  [TOOL] 'GPT'
  [TOOL] 'SQL'
  [SKILL] 'data engineering'
  [SKILL] 'statistical modeling'
  [SKILL] 'A/B testing'
  [SKILL] 'MLOps practices'


## 6. Run Extraction — German Job

In [27]:
print("Running LangExtract on German job posting...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_de = lx.extract(
    text_or_documents=german_job,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434",
    fence_output=False,
    use_schema_constraints=False,
)

print("=== GERMAN JOB — Extracted Entities ===")
for extraction in result_de.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

Running LangExtract on German job posting...
(This may take 30-60 seconds with Llama 3.1 8B)



LangExtract: Processing, current=431 chars, processed=0 chars:  [00:20]

=== GERMAN JOB — Extracted Entities ===
  [SKILL] 'Machine Learning'
  [SKILL] 'Datenanalyse'
  [SKILL] 'MLOps'
  [SKILL] 'statistische Modellierung'
  [SKILL] 'A/B-Testing'
  [TOOL] 'Python'
  [TOOL] 'PyTorch'
  [TOOL] 'TensorFlow'
  [TOOL] 'AWS'
  [TOOL] 'Azure'
  [TOOL] 'GCP'
  [TOOL] 'Docker'
  [TOOL] 'Kubernetes'
  [SKILL] 'NLP'
  [SKILL] 'Transformer-Modelle'
  [TOOL] 'SQL'


In [40]:
print("Running LangExtract on description...")
print("(This may take 30-60 seconds with Llama 3.1 8B)\n")

result_des = lx.extract(
    text_or_documents=description,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://host.docker.internal:11434",
    fence_output=False,
    use_schema_constraints=False,
)

print("=== description — Extracted Entities ===")
for extraction in result_des.extractions:
    print(f"  [{extraction.extraction_class}] '{extraction.extraction_text}'")

Running LangExtract on description...
(This may take 30-60 seconds with Llama 3.1 8B)



LangExtract: Processing, current=654 chars, processed=0 chars:  [00:26]

=== description — Extracted Entities ===
  [SKILL] 'Automatisierung von ML-Pipelines'
  [SKILL] 'Plattformbetrieb und Orchestrierung'
  [SKILL] 'Optimierung von Monitoring- und Logging-Lösungen'
  [SKILL] 'Infrastructure-as-Code'
  [SKILL] 'CI/CD-Prozessen'
  [SKILL] 'Versionsverwaltung'
  [SKILL] 'GPU-Integration'
  [TOOL] 'Ansible'
  [TOOL] 'Terraform'
  [TOOL] 'Git'
  [TOOL] 'MLflow'
  [TOOL] 'Kubeflow'
  [TOOL] 'Airflow'
  [TOOL] 'Grafana'
  [TOOL] 'Kibana'
  [TOOL] 'Kubernetes'
  [TOOL] 'OpenShift'
  [TOOL] 'NVIDIA'


## 7. Compare Side by Side
Both jobs describe the same role — do the extractions match?

In [19]:
en_skills = [e.extraction_text for e in result_en.extractions if e.extraction_class == "SKILL"]
en_tools  = [e.extraction_text for e in result_en.extractions if e.extraction_class == "TOOL"]
de_skills = [e.extraction_text for e in result_de.extractions if e.extraction_class == "SKILL"]
de_tools  = [e.extraction_text for e in result_de.extractions if e.extraction_class == "TOOL"]

print("ENGLISH")
print(f"  SKILLs: {en_skills}")
print(f"  TOOLs:  {en_tools}")
print()
print("GERMAN")
print(f"  SKILLs: {de_skills}")
print(f"  TOOLs:  {de_tools}")
print()
print(f"English total: {len(en_skills) + len(en_tools)} entities")
print(f"German total:  {len(de_skills) + len(de_tools)} entities")

ENGLISH
  SKILLs: ['machine learning', 'data science', 'NLP techniques', 'transformer-based models', 'data engineering', 'statistical modeling', 'A/B testing', 'MLOps practices']
  TOOLs:  ['Python', 'PyTorch', 'TensorFlow', 'AWS', 'GCP', 'Azure', 'Docker', 'Kubernetes', 'CI/CD pipelines', 'BERT', 'GPT', 'SQL']

GERMAN
  SKILLs: ['Machine Learning', 'Datenanalyse', 'MLOps', 'statistische Modellierung', 'A/B-Testing', 'NLP', 'Transformer-Modelle']
  TOOLs:  ['Python', 'PyTorch', 'TensorFlow', 'AWS', 'Azure', 'GCP', 'Docker', 'Kubernetes', 'SQL']

English total: 20 entities
German total:  16 entities


## 8. Check Source Grounding
LangExtract's key feature — every extraction is mapped back to its exact position in the text.

In [20]:
print("=== Source Grounding (English) ===")
print("Each extraction shows where in the text it was found.\n")

for e in result_en.extractions:
    # char_start and char_end show exact position in source text
    has_grounding = hasattr(e, 'char_start') and e.char_start is not None
    if has_grounding:
        snippet = english_job[e.char_start:e.char_end]
        print(f"  [{e.extraction_class}] '{e.extraction_text}' @ chars {e.char_start}-{e.char_end}")
        print(f"    Source text: '...{snippet}...'")
    else:
        print(f"  [{e.extraction_class}] '{e.extraction_text}' (no grounding available with this model)")

=== Source Grounding (English) ===
Each extraction shows where in the text it was found.

  [SKILL] 'machine learning' (no grounding available with this model)
  [SKILL] 'data science' (no grounding available with this model)
  [TOOL] 'Python' (no grounding available with this model)
  [TOOL] 'PyTorch' (no grounding available with this model)
  [TOOL] 'TensorFlow' (no grounding available with this model)
  [TOOL] 'AWS' (no grounding available with this model)
  [TOOL] 'GCP' (no grounding available with this model)
  [TOOL] 'Azure' (no grounding available with this model)
  [TOOL] 'Docker' (no grounding available with this model)
  [TOOL] 'Kubernetes' (no grounding available with this model)
  [TOOL] 'CI/CD pipelines' (no grounding available with this model)
  [SKILL] 'NLP techniques' (no grounding available with this model)
  [SKILL] 'transformer-based models' (no grounding available with this model)
  [TOOL] 'BERT' (no grounding available with this model)
  [TOOL] 'GPT' (no grounding 

## 10. Gemini Test

End-to-end run on a real job posting from the dataset:
1. Load API key from `.env`
2. Pick first job with a non-empty description from `data/processed/jobs_combined_clean.json`
3. Extract requirements section using Gemini (same prompt as `scripts/07_test_section_extraction.py`)
4. Run LangExtract entity extraction with `gemini-2.0-flash`

In [51]:
import json
import os
import textwrap
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.environ["LANGEXTRACT_API_KEY"]
print("✅ API key loaded")

# Load first job with a non-empty description
with open("../../data/processed/jobs_combined_clean.json", encoding="utf-8") as f:
    all_jobs = json.load(f)

job = next(
    j for j in all_jobs
    if j.get("description_clean") and len(j["description_clean"]) > 100
)

print(f"Job title:   {job.get('title', 'N/A')}")
print(f"Company:     {job.get('companyName', 'N/A')}")
print(f"Language:    {job.get('language', 'N/A')}")
print(f"Description: {len(job['description_clean'])} chars")

✅ API key loaded
Job title:   Data Scientist - (Logistics, Seamless Deliveries)
Company:     Delivery Hero
Language:    English
Description: 5971 chars


In [52]:
from google import genai as google_genai
import os

client = google_genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Available Gemini models that support generateContent:\n")
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(f"  {m.name}")

Available Gemini models that support generateContent:

  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.0-flash
  models/gemini-2.0-flash-001
  models/gemini-2.0-flash-lite-001
  models/gemini-2.0-flash-lite
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-3-1b-it
  models/gemma-3-4b-it
  models/gemma-3-12b-it
  models/gemma-3-27b-it
  models/gemma-3n-e4b-it
  models/gemma-3n-e2b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-2.5-flash-lite-preview-09-2025
  models/gemini-3-pro-preview
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3-pro-image-preview
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/gemini-robotics-er-1.5-preview
  models/gemini-2.5-computer-

In [53]:
!pip install google-genai -q --break-system-packages

In [ ]:
import langextract as lx

# Use full description directly (section extraction happens in the production script)
requirements_section = job["description_clean"]

print("Running LangExtract with gemini-2.5-flash...\n")

result_gemini = lx.extract(
    text_or_documents=requirements_section,
    prompt_description=prompt,
    examples=examples,
    model_id="gemini-2.5-flash",
    fence_output=True,
    use_schema_constraints=True,
)

skills = [e for e in result_gemini.extractions if e.extraction_class == "SKILL"]
tools  = [e for e in result_gemini.extractions if e.extraction_class == "TOOL"]

print("=== Extracted Entities ===")
for e in result_gemini.extractions:
    print(f"  [{e.extraction_class}] {e.extraction_text}")

print(f"\n=== Summary ===")
print(f"  Total entities : {len(result_gemini.extractions)}")
print(f"  SKILLs         : {len(skills)}")
print(f"  TOOLs          : {len(tools)}")

In [64]:
from google import genai
import os
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

## 9. Observations

Fill this in after running:

- Did LangExtract extract the expected entities?
- Did it handle German correctly?
- Were there false positives (wrong entities)?
- Were there false negatives (missed entities)?
- Did source grounding work?
- How does this compare to GLiNER (zero-shot, F1=0.27) in feel?

Key difference from your original Llama annotation:  
LangExtract structures the output AND grounds it to source positions automatically,  
whereas your original `08_annotate_llm.py` required manual span-mapping (the step that failed ~15% of the time).